In [21]:
from typing import Any

import requests

# Define the API endpoint and parameters
# This example searches for studies related to 'asthma'
url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "pageSize": 1          # Number of results per page
}

response = requests.get(url, params=params)
response.raise_for_status()  # Check for HTTP errors

data = response.json()# Print the retrieved data
page_studies = data.get("studies", [])
protocol: dict = page_studies[0].get("protocolSection", {})

def _get(obj: dict, *keys: str, default: Any = None) -> Any:
    """Safe nested dict getter; returns *default* if any key is missing."""
    cur = obj
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key, default)
        if cur is None:
            return default
    return cur

In [22]:
from datetime import date
import re
from typing import Optional

from dateutil import parser as du_parser

_YYYY_MM_RE = re.compile(r"^\d{4}-\d{2}$")
_YYYY_RE = re.compile(r"^\d{4}$")

def _parse_date(value: Any) -> Optional[date]:
    """
    Robustly parse a date value from the API.

    Handles:
      "2024-01-15"          → date(2024, 1, 15)
      "2024-01"             → date(2024, 1, 1)   (first of month)
      "2024"                → date(2024, 1, 1)   (first of year)
      "January 15, 2024"    → date(2024, 1, 15)  (via dateutil)
      {"date": "2024-01-15", ...}                (struct form from API)
      None / ""             → None
    """
    if value is None:
        return None

    # The API sometimes wraps dates in a struct: {"date": "YYYY-MM-DD", "type": "..."}
    if isinstance(value, dict):
        value = value.get("date")
    if not value:
        return None

    s = str(value).strip()

    if _YYYY_MM_RE.match(s):
        return date.fromisoformat(f"{s}-01")
    if _YYYY_RE.match(s):
        return date(int(s), 1, 1)

    try:
        return du_parser.parse(s).date()
    except (ValueError, TypeError):
        return None

In [23]:
for chunk_idx, raw_chunk in enumerate(page_studies):
    protocol: dict = raw_chunk.get("protocolSection", {})
    id_mod = protocol.get("identificationModule", {})
    nct_id: str = id_mod.get("nctId")
    status_mod = protocol.get("statusModule", {})
    start_date = _parse_date(_get(status_mod, "startDateStruct"))
    print(f"NCT ID: {nct_id}, Start Date: {start_date}")

NCT ID: NCT07157735, Start Date: 2026-02-02


In [11]:
import httpx
import ssl
import certifi

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}
context = ssl.create_default_context()
context.load_verify_locations(certifi.where())
with httpx.Client(
    headers={
        "User-Agent": "python-requests/2.31.0",
        "Accept-Encoding": "gzip, deflate",  # no 'br'
        "Accept": "*/*",
        "Connection": "keep-alive",
    },
    http2=False,
) as client:
    response = client.get(url, params=params, headers=headers)
    print(response)  # Should now be 200

<Response [403 Forbidden]>


In [57]:
"""
Transform task: map raw clinicalTrials.gov v2 JSON → ClinicalTrial models.

The v2 API returns studies as a nested ``protocolSection`` object.
Each helper below extracts one slice of that object and maps it to the
flat BigQuery column names defined in ``models/trial.py``.

Bad records (missing nct_id or unparseable startDate) are skipped and
logged rather than raising, so a single malformed study never aborts a
full-load run.
"""
from __future__ import annotations

import datetime
import json
import logging
import re
from datetime import date
from typing import Any, Optional

from dateutil import parser as du_parser

from src.models.trial import  Intervention, Outcome, Reference

logger = logging.getLogger(__name__)


# ── Date parsing ───────────────────────────────────────────────────────────────

_YYYY_MM_RE = re.compile(r"^\d{4}-\d{2}$")
_YYYY_RE = re.compile(r"^\d{4}$")


def _parse_date(value: Any) -> Optional[date]:
    """
    Robustly parse a date value from the API.

    Handles:
      "2024-01-15"          → date(2024, 1, 15)
      "2024-01"             → date(2024, 1, 1)   (first of month)
      "2024"                → date(2024, 1, 1)   (first of year)
      "January 15, 2024"    → date(2024, 1, 15)  (via dateutil)
      {"date": "2024-01-15", ...}                (struct form from API)
      None / ""             → None
    """
    if value is None:
        return None

    # The API sometimes wraps dates in a struct: {"date": "YYYY-MM-DD", "type": "..."}
    if isinstance(value, dict):
        value = value.get("date")
    if not value:
        return None

    s = str(value).strip()

    if _YYYY_MM_RE.match(s):
        return date.fromisoformat(f"{s}-01")
    if _YYYY_RE.match(s):
        return date(int(s), 1, 1)

    try:
        return du_parser.parse(s).date()
    except (ValueError, TypeError):
        logger.debug("Unparseable date string: %r", s)
        return None


# ── Section extractors ─────────────────────────────────────────────────────────

def _get(obj: dict, *keys: str, default: Any = None) -> Any:
    """Safe nested dict getter; returns *default* if any key is missing."""
    cur = obj
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key, default)
        if cur is None:
            return default
    return cur


def _extract_references(protocol: dict):
    raw = _get(protocol, "referencesModule", "references", default=[])
    refs = []
    for r in raw or []:
        pmid_raw = r.get("pmid")
        try:
            pmid = int(pmid_raw) if pmid_raw is not None else None
        except (ValueError, TypeError):
            pmid = None
        refs.append(
            {
                'citation': r.get("citation"),
                'type': r.get("type"),
                'pmid': pmid,
            }
        )
    return refs


def _extract_primary_outcomes(protocol: dict):
    raw = _get(protocol, "outcomesModule", "primaryOutcomes", default=[])
    return [
        {
            'measure': o.get("measure"),
            'description': o.get("description"),
            'timeFrame': o.get("timeFrame"),
        }
        for o in (raw or [])
    ]


def _extract_secondary_outcomes(protocol: dict):
    raw = _get(protocol, "outcomesModule", "secondaryOutcomes", default=[])
    return [
        {
            'measure': o.get("measure"),
            'description': o.get("description"),
            'timeFrame': o.get("timeFrame"),
        }
        for o in (raw or [])
    ]


def _extract_interventions(protocol: dict):
    raw = _get(protocol, "armsInterventionsModule", "interventions", default=[])
    return [
        {
            'type': i.get("type"),
            'name': i.get("name"),
            'description': i.get("description"),
        }
        for i in (raw or [])
    ]


def _extract_collaborator_names(protocol: dict) -> list[str]:
    collaborators = _get(protocol, "sponsorCollaboratorsModule", "collaborators", default=[])
    return [c["name"] for c in (collaborators or []) if c.get("name")]


# ── Main transform ─────────────────────────────────────────────────────────────

def transform_study(raw_study: dict):
    """
    Map one raw API study dict to a ``ClinicalTrial`` model.

    Returns ``None`` (and logs a warning) for studies that are missing
    required fields (``nctId`` or ``startDate``).
    """
    protocol: dict = raw_study.get("protocolSection", {})

    # ── Identification ────────────────────────────────────────────────────────
    id_mod = protocol.get("identificationModule", {})
    nct_id: Optional[str] = id_mod.get("nctId")
    if not nct_id:
        logger.warning("Skipping study with no nctId: %s", str(raw_study)[:120])
        return None

    # ── Status / Dates ────────────────────────────────────────────────────────
    status_mod = protocol.get("statusModule", {})
    start_date = _parse_date(_get(status_mod, "startDateStruct"))
    logger.info("Parsed start date for %s: %s", nct_id, start_date)
    if start_date is None:
        logger.warning("Skipping %s: missing/unparseable startDate", nct_id)
        return None

    last_update = _parse_date(_get(status_mod, "lastUpdatePostDateStruct"))
    if last_update is None:
        # Fall back to today so the NOT NULL constraint is satisfied
        from datetime import date as _date
        last_update = _date.today()
        logger.debug("%s: no lastUpdatePostDate, defaulting to today", nct_id)

    # ── Sponsor / Collaborators ───────────────────────────────────────────────
    sponsor_mod = protocol.get("sponsorCollaboratorsModule", {})
    responsible_party = sponsor_mod.get("responsibleParty", {})

    # ── Design ────────────────────────────────────────────────────────────────
    design_mod = protocol.get("designModule", {})
    design_info = design_mod.get("designInfo", {})
    masking_info = design_info.get("maskingInfo", {})
    enrollment_info = design_mod.get("enrollmentInfo", {})

    # ── Eligibility ───────────────────────────────────────────────────────────
    elig_mod = protocol.get("eligibilityModule", {})

    # ── Description ───────────────────────────────────────────────────────────
    desc_mod = protocol.get("descriptionModule", {})

    return {
    "nct_id": nct_id,
    "lastUpdatePostDate": last_update,
    "startDate": start_date,
    "completionDate": _parse_date(_get(status_mod, "completionDateStruct")),
    "primaryCompletionDate": _parse_date(_get(status_mod, "primaryCompletionDateStruct")),
    "overallStatus": status_mod.get("overallStatus"),

    "officialTitle": id_mod.get("officialTitle"),
    "organization": _get(id_mod, "organization", "fullName"),

    "leadSponsorName": _get(sponsor_mod, "leadSponsor", "name"),
    "leadSponsorCollaborators": _extract_collaborator_names(protocol),
    "investigatorFullName": responsible_party.get("investigatorFullName"),

    "conditionsModule": _get(protocol, "conditionsModule", "conditions", default=[]) or [],

    "designModule_studyType": design_mod.get("studyType"),
    "designModule_designInfo_allocation": design_info.get("allocation"),
    "designModule_designInfo_interventionModel": design_info.get("interventionModel"),
    "designModule_designInfo_primaryPurpose": design_info.get("primaryPurpose"),
    "designModule_designInfo_maskingInfo_masking": masking_info.get("masking"),
    "designModule_enrollmentInfo_count": enrollment_info.get("count"),

    "armsInterventionsModule_interventions": _extract_interventions(protocol),
    "outcomesModule_primaryOutcomes": _extract_primary_outcomes(protocol),
    "outcomesModule_secondaryOutcomes": _extract_secondary_outcomes(protocol),

    "eligibilityCriteria": elig_mod.get("eligibilityCriteria"),
    "eligibilityCriteria_minimumAge": elig_mod.get("minimumAge"),
    "eligibilityCriteria_sex": elig_mod.get("sex"),
    "eligibilityCriteria_stdAges": elig_mod.get("stdAges") or [],
    "eligibilityCriteria_healthyVolunteers": elig_mod.get("healthyVolunteers"),

    "detailedDescription": desc_mod.get("detailedDescription"),
    "references": _extract_references(protocol),
}

def json_serial(obj):
    """JSON serializer for objects not serializable by default json code"""
    if isinstance(obj, (datetime.datetime, datetime.date)):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def transform_studies(raw_studies: list[dict]) -> list[dict]:
    """
    Transform a list of raw API study dicts.  Invalid records are
    dropped; the count of dropped records is logged for observability.
    """
    trials: list= []
    dropped = 0
    for raw in raw_studies:
        trial = transform_study(raw)
        if trial is not None:
            trials.append(trial)
        else:
            dropped += 1

    print(
        "Transform complete: %d valid, %d dropped (%.1f%%)",
        len(trials),
        dropped,
        100 * dropped / max(len(raw_studies), 1),
    )
    #print(json.dumps(trials[0], default=json_serial)) # Debug: print first transformed trial
    return trials


In [62]:
# Struct fields: default keys so PyArrow never sees a partial dict
_STRUCT_DEFAULTS: dict[str, dict] = {
    "references": {
        "citation": None, "type": None, "pmid": None,
    },
    "outcomesModule_primaryOutcomes": {
        "timeFrame": None, "description": None, "measure": None,
    },
    "outcomesModule_secondaryOutcomes": {
        "timeFrame": None, "description": None, "measure": None,
    },
    "armsInterventionsModule_interventions": {
        "description": None, "type": None, "name": None,
    },
}

# Plain list fields that must default to [] when absent/None
_LIST_FIELDS = ("leadSponsorCollaborators", "conditionsModule", "eligibilityCriteria_stdAges")
def _coerce_record(record: dict) -> dict:
    """
    Normalise a raw trial dict so every field matches PARQUET_SCHEMA:
      - Struct list fields : fill missing keys with None
      - Plain list fields  : default to [] when absent/None
      - Everything else    : pass through (PyArrow maps None → null)
    """
    coerced = dict(record)  # shallow copy — never mutate the caller's data
 
    for field, defaults in _STRUCT_DEFAULTS.items():
        raw = coerced.get(field) or []
        coerced[field] = [{**defaults, **item} for item in raw]
 
    for field in _LIST_FIELDS:
        if coerced.get(field) is None:
            coerced[field] = []
 
    return coerced

In [63]:
import pyarrow as pa
import pyarrow.parquet as pq
import dask.dataframe as dd

print(len(page_studies))

# Define schema with nested types
PARQUET_SCHEMA = pa.schema([
    pa.field("references", pa.list_(
        pa.struct([
            pa.field("citation", pa.string()),
            pa.field("type", pa.string()),
            pa.field("pmid", pa.int64())
        ])
    )),

    pa.field("lastUpdatePostDate", pa.date32()),
    pa.field("startDate", pa.date32()),
    pa.field("completionDate", pa.date32()),
    pa.field("primaryCompletionDate", pa.date32()),

    pa.field("eligibilityCriteria_minimumAge", pa.string()),
    pa.field("organization", pa.string()),
    pa.field("overallStatus", pa.string()),

    pa.field("outcomesModule_primaryOutcomes", pa.list_(
        pa.struct([
            pa.field("timeFrame", pa.string()),
            pa.field("description", pa.string()),
            pa.field("measure", pa.string())
        ])
    )),

    pa.field("armsInterventionsModule_interventions", pa.list_(
        pa.struct([
            pa.field("description", pa.string()),
            pa.field("type", pa.string()),
            pa.field("name", pa.string())
        ])
    )),

    pa.field("designModule_enrollmentInfo_count", pa.int64()),
    pa.field("eligibilityCriteria_stdAges", pa.large_list(pa.string())),
    pa.field("designModule_designInfo_maskingInfo_masking", pa.string()),
    pa.field("designModule_designInfo_interventionModel", pa.string()),
    pa.field("conditionsModule", pa.list_(pa.string())),
    pa.field("eligibilityCriteria", pa.string()),
    pa.field("designModule_designInfo_primaryPurpose", pa.string()),
    pa.field("detailedDescription", pa.string()),
    pa.field("leadSponsorCollaborators", pa.list_(pa.string())),
    pa.field("investigatorFullName", pa.string()),
    pa.field("eligibilityCriteria_sex", pa.string()),
    pa.field("designModule_designInfo_allocation", pa.string()),
    pa.field("officialTitle", pa.string()),
    pa.field("eligibilityCriteria_healthyVolunteers", pa.bool_()),

    pa.field("outcomesModule_secondaryOutcomes", pa.list_(
        pa.struct([
            pa.field("timeFrame", pa.string()),
            pa.field("description", pa.string()),
            pa.field("measure", pa.string())
        ])
    )),

    pa.field("designModule_studyType", pa.string()),
    pa.field("leadSponsorName", pa.string()),
    pa.field("nct_id", pa.string()),
])

# ─────────────────────────────────────────────────────────────────────────────
# 4. CONVERT LIST OF DICTS → PYARROW TABLE
# ─────────────────────────────────────────────────────────────────────────────
 
def records_to_arrow_table(records: list[dict]) -> pa.Table:
    """
    Convert a batch of raw trial dicts to a typed PyArrow Table.
    Builds column-by-column so each array is cast to its declared schema type.
    """
    coerced = [_coerce_record(r) for r in records]
 
    arrays = {
        field.name: pa.array(
            [row.get(field.name) for row in coerced],
            type=field.type,
        )
        for field in PARQUET_SCHEMA
    }
 
    return pa.table(arrays, schema=PARQUET_SCHEMA)



1


In [66]:

table = records_to_arrow_table(transform_studies(page_studies))  # your dict
from src.config import settings
output_path = f"gs://{settings.gcs_bucket}/{settings.gcs_prefix}/{1}/chunk_{1}.parquet"

Transform complete: %d valid, %d dropped (%.1f%%) 1 0 0.0


In [ ]:
import pandas as pd
# 1. Convert your list of records to a Dask DataFrame
# (Dask uses Pandas under the hood for this step)
pdf = pd.DataFrame(data)
dask_df = dd.from_pandas(pdf, npartitions=1)
# Wrap in list (PyArrow expects a list of records)
table = pa.Table.from_pylist(data, schema=PARQUET_SCHEMA)

# Write to Parquet
#pq.write_table(table, "clinical_trial.parquet")

# Read back with Dask
#dask_df = dd.read_parquet("clinical_trial.parquet")
#print(dask_df.compute())

# Access nested columns
#print(dask_df["armsInterventionsModule_interventions"].compute())
#print(dask_df["eligibilityCriteria_stdAges"].compute())


In [68]:
import pyarrow.parquet as pq
# 2. Write directly to GCS
pq.write_table(
    table,
    output_path,
    compression='snappy'             # Standard efficient compression
)

In [70]:
from google.cloud import bigquery
BQ_SCHEMA = [
    bigquery.SchemaField("references", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("citation", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("type",     "STRING", mode="NULLABLE"),
        bigquery.SchemaField("pmid",     "INT64",  mode="NULLABLE"),
    ]),
    bigquery.SchemaField("lastUpdatePostDate",    "DATE", mode="REQUIRED"),
    bigquery.SchemaField("startDate",             "DATE", mode="REQUIRED"),
    bigquery.SchemaField("completionDate",        "DATE", mode="NULLABLE"),
    bigquery.SchemaField("primaryCompletionDate", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("eligibilityCriteria_minimumAge",              "STRING", mode="NULLABLE"),
    bigquery.SchemaField("organization",                                "STRING", mode="NULLABLE"),
    bigquery.SchemaField("overallStatus",                               "STRING", mode="NULLABLE"),
    bigquery.SchemaField("outcomesModule_primaryOutcomes", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("timeFrame",   "STRING", mode="NULLABLE"),
        bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("measure",     "STRING", mode="NULLABLE"),
    ]),
    bigquery.SchemaField("armsInterventionsModule_interventions", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("type",        "STRING", mode="NULLABLE"),
        bigquery.SchemaField("name",        "STRING", mode="NULLABLE"),
    ]),
    bigquery.SchemaField("designModule_enrollmentInfo_count",           "INT64",  mode="NULLABLE"),
    bigquery.SchemaField("eligibilityCriteria_stdAges",                 "STRING", mode="REPEATED"),
    bigquery.SchemaField("designModule_designInfo_maskingInfo_masking", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("designModule_designInfo_interventionModel",   "STRING", mode="NULLABLE"),
    bigquery.SchemaField("conditionsModule",                            "STRING", mode="REPEATED"),
    bigquery.SchemaField("eligibilityCriteria",                         "STRING", mode="NULLABLE"),
    bigquery.SchemaField("designModule_designInfo_primaryPurpose",      "STRING", mode="NULLABLE"),
    bigquery.SchemaField("detailedDescription",                         "STRING", mode="NULLABLE"),
    bigquery.SchemaField("leadSponsorCollaborators",                    "STRING", mode="REPEATED"),
    bigquery.SchemaField("investigatorFullName",                        "STRING", mode="NULLABLE"),
    bigquery.SchemaField("eligibilityCriteria_sex",                     "STRING", mode="NULLABLE"),
    bigquery.SchemaField("designModule_designInfo_allocation",          "STRING", mode="NULLABLE"),
    bigquery.SchemaField("officialTitle",                               "STRING", mode="NULLABLE"),
    bigquery.SchemaField("eligibilityCriteria_healthyVolunteers",       "BOOL",   mode="NULLABLE"),
    bigquery.SchemaField("outcomesModule_secondaryOutcomes", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("timeFrame",   "STRING", mode="NULLABLE"),
        bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("measure",     "STRING", mode="NULLABLE"),
    ]),
    bigquery.SchemaField("designModule_studyType", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("leadSponsorName",        "STRING", mode="NULLABLE"),
    bigquery.SchemaField("nct_id",                 "STRING", mode="REQUIRED"),
]

In [59]:
    # 2. Use storage_options for the project ID
from src.config import settings

output_path = f"gs://{settings.gcs_bucket}/{settings.gcs_prefix}/{1}/chunk_{1}.parquet"

dask_df.to_parquet(
        output_path,
        #engine="pyarrow",
        write_index=False,
        schema=PARQUET_SCHEMA,
        # This creates the '_metadata' file BigQuery loves
        write_metadata_file=True,
        overwrite=True,
        #use_compliant_nested_type=True,
        # MUST provide project info for GCS access
        storage_options={"project": settings.gcp_project_id}
    )

ValueError: Failed to convert partition to expected pyarrow schema:
    `ArrowNotImplementedError('Unsupported cast from large_string to list using function cast_list', 'Conversion failed for column references with type string')`

Expected partition schema:
    references: list<item: struct<citation: string, type: string, pmid: int64>>
      child 0, item: struct<citation: string, type: string, pmid: int64>
          child 0, citation: string
          child 1, type: string
          child 2, pmid: int64
    lastUpdatePostDate: date32[day]
    startDate: date32[day]
    completionDate: date32[day]
    primaryCompletionDate: date32[day]
    eligibilityCriteria_minimumAge: string
    organization: string
    overallStatus: string
    outcomesModule_primaryOutcomes: list<item: struct<timeFrame: string, description: string, measure: string>>
      child 0, item: struct<timeFrame: string, description: string, measure: string>
          child 0, timeFrame: string
          child 1, descripti

In [ ]:
from google.cloud import bigquery, storage
bq_client = bigquery.Client(project=settings.gcp_project_id)
table_ref = bigquery.TableReference.from_string(settings.bq_full_table_id)
parquet_options = bigquery.ParquetOptions()
parquet_options.enable_list_inference = True
job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        schema=BQ_SCHEMA,
        # Correctly maps Parquet repeated fields to BQ ARRAY / RECORD
        parquet_options=parquet_options,
    )
wildcard_uri = output_path = f"gs://{settings.gcs_bucket}/{settings.gcs_prefix}/{1}/*.parquet"
load_job = bq_client.load_table_from_uri(
        wildcard_uri, table_ref, job_config=job_config
    )
load_job.result()

BadRequest: 400 Error while reading data, error message: Schema mismatch: referenced variable 'armsInterventionsModule_interventions.list.element.type' has array levels of 1, while the corresponding field path to Parquet column has 0 repeated fields File: gs://bitziv_health_data/clinical_trials_gov_stg/1/chunk_1.parquet; reason: invalid, location: gs://bitziv_health_data/clinical_trials_gov_stg/1/chunk_1.parquet, message: Error while reading data, error message: Schema mismatch: referenced variable 'armsInterventionsModule_interventions.list.element.type' has array levels of 1, while the corresponding field path to Parquet column has 0 repeated fields File: gs://bitziv_health_data/clinical_trials_gov_stg/1/chunk_1.parquet